# credit-risk-xai-framework

**Author:** Sana Ur Rehman  
**Profession:** Data Scientist  
**Created:** 2026  

---

## License

This project is licensed under the **MIT License**. 

You are free to use, modify, distribute, and build upon this work for both commercial and non-commercial purposes, provided you give appropriate **credit** to the original author. For the full legal text and conditions, please refer to the `LICENSE` file included in this project's repository.

---

## Citation

If you reference or build upon this project, please provide appropriate credit.

For formal citation information, please see the project's `README.md` and `CITATION.cff` files.

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------

In [2]:
df = pd.read_csv('data/loans_full_schema.csv')

In [3]:
print("="*50)
print("COMPREHENSIVE DATASET OVERVIEW")
print("="*50)

# Basic Info
print("Dataset Shape:", df.shape)
print(f"Total cells: {df.size:,}")
print(f"Total memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "="*30)
print("DATA TYPES & STRUCTURE")
print("="*30)
print(df.dtypes)
print(f"\nData type distribution:")
print(df.dtypes.value_counts())

# Detailed info
print("\n" + "="*30)
print("DETAILED COLUMN INFO")
print("="*30)
df.info()

print("\n" + "="*30)
print("MISSING VALUES ANALYSIS")
print("="*30)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])  # Only show columns with missing values

print("\n" + "="*30)
print("UNIQUE VALUES COUNT")
print("="*30)
print(df.nunique().sort_values(ascending=False))

print("\n" + "="*30)
print("STATISTICAL SUMMARY")
print("="*30)
print(df.describe(include='all'))

# Additional insights for categorical columns
print("\n" + "="*30)
print("CATEGORICAL COLUMNS PREVIEW")
print("="*30)
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols[:5]:  # Show first 5 categorical columns
    print(f"\n{col}:")
    print(df[col].value_counts().head())

# Numerical columns distribution
print("\n" + "="*30)
print("NUMERICAL COLUMNS DISTRIBUTION")
print("="*30)
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in numerical_cols:
    print(f"\n{col}: Min={df[col].min()}, Max={df[col].max()}, "
          f"Median={df[col].median()}, Skew={df[col].skew():.2f}")

print("\n" + "="*30)
print("DUPLICATE ROWS")
print("="*30)
duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {duplicates}")
if duplicates > 0:
    print(f"Percentage of duplicates: {(duplicates/len(df))*100:.2f}%")

COMPREHENSIVE DATASET OVERVIEW
Dataset Shape: (10000, 55)
Total cells: 550,000
Total memory usage: 9.96 MB

DATA TYPES & STRUCTURE
emp_title                            object
emp_length                          float64
state                                object
homeownership                        object
annual_income                       float64
verified_income                      object
debt_to_income                      float64
annual_income_joint                 float64
verification_income_joint            object
debt_to_income_joint                float64
delinq_2y                             int64
months_since_last_delinq            float64
earliest_credit_line                  int64
inquiries_last_12m                    int64
total_credit_lines                    int64
open_credit_lines                     int64
total_credit_limit                    int64
total_credit_utilized                 int64
num_collections_last_12m              int64
num_historical_failed_to_pay     

# ---------------------------------------------------------
# 2. TARGET VARIABLE DEFINITION
# ---------------------------------------------------------
We treat 'Current' and 'Fully Paid' as Good (0).

We treat 'Late', 'In Grace Period', and 'Charged Off' as Bad/Risk (1).

In [4]:
def create_target(status):
    if status in ['Current', 'Fully Paid']:
        return 0
    else:
        return 1

df['loan_status_binary'] = df['loan_status'].apply(create_target)
print("\nTarget Distribution (0 = Good, 1 = Risk):")
print(df['loan_status_binary'].value_counts())


Target Distribution (0 = Good, 1 = Risk):
loan_status_binary
0    9822
1     178
Name: count, dtype: int64


# ---------------------------------------------------------
# 3. HANDLE MISSING VALUES
# ---------------------------------------------------------

Fill Joint Application columns with 0 (implies Individual application)

In [5]:
joint_cols = ['annual_income_joint', 'debt_to_income_joint']
for col in joint_cols:
    df[col] = df[col].fillna(0)

Fill Verification Joint with 'Not Applicable'

In [6]:
df['verification_income_joint'] = df['verification_income_joint'].fillna('Not Applicable')

Fill Employment gaps with 0 or 'Unknown'

In [7]:
df['emp_length'] = df['emp_length'].fillna(0)
df['emp_title'] = df['emp_title'].fillna('Unknown')

Fill Delinquency history: Missing means "No Delinquency" -> 999 (safe value for trees)

In [8]:
df['months_since_last_delinq'] = df['months_since_last_delinq'].fillna(999)
df['months_since_90d_late'] = df['months_since_90d_late'].fillna(999)
df['months_since_last_credit_inquiry'] = df['months_since_last_credit_inquiry'].fillna(0)

Drop small number of rows missing critical 'debt_to_income' data (only ~24 rows)

In [9]:
df = df.dropna(subset=['debt_to_income'])

# ---------------------------------------------------------
# 4. FINAL CLEANUP
# ---------------------------------------------------------

Drop high cardinality column not useful for this specific model

In [10]:
if 'emp_title' in df.columns:
    df = df.drop(columns=['emp_title'])

Save to CSV

In [11]:
df.to_csv('data/clean_data.csv', index=False)
print("\nSUCCESS: 'clean_data.csv' has been saved!")


SUCCESS: 'clean_data.csv' has been saved!
